In [44]:
import os, glob, time, datetime, json, csv, logging
import pypdf, pytesseract
from typing import Literal, Optional, Any
from pydantic import BaseModel, Field
from sqlalchemy import create_engine, Column, String, Float, Integer, Text, DateTime
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
from pdf2image import convert_from_path
from litellm import completion
from litellm.exceptions import RateLimitError
from dotenv import load_dotenv
import pytesseract
import pandas as pd

load_dotenv()
Base = declarative_base()
logging.getLogger("pypdf").setLevel(logging.ERROR)

# --- MODELS FOR AI VALIDATION ---
class DocReport(BaseModel):
    category: Literal["Cease", "Irrelevant", "Uncertain"]
    confidence: float
    reasoning: str

class ExtractionReport(BaseModel):
    customer_name: str
    date_on_document: Optional[str] = "N/A"
    account_id: Optional[Any] = "N/A"
    summary: str

# --- DATABASE SCHEMA ---
class LegalAuditDB(Base):
    __tablename__ = 'cease_document_logs'
    id = Column(Integer, primary_key=True)
    file_name = Column(String, unique=True)
    category = Column(String)
    confidence = Column(Float)
    customer_name = Column(String)
    extracted_date = Column(String)
    account_id = Column(String)
    reasoning = Column(Text)
    processed_at = Column(DateTime, default=datetime.datetime.utcnow)
    needs_review = Column(Integer, default=0)

C:\Users\Geeta Madhuri\AppData\Local\Temp\ipykernel_4956\1482634804.py:16: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [45]:
import json
import time

# --- CONFIGURATION & MODELS ---
PRIMARY_MODEL = "groq/llama-3.3-70b-versatile"
INSTANT_MODEL = "groq/llama-3.1-8b-instant" 

class DatabaseAgent:
    def __init__(self, db_url='sqlite:///legal_docs.db'):
        self.engine = create_engine(db_url)
        Base.metadata.create_all(self.engine)
        self.Session = sessionmaker(bind=self.engine)

    def is_already_processed(self, file_name):
        session = self.Session()
        exists = session.query(LegalAuditDB).filter_by(file_name=file_name).first() is not None
        session.close()
        return exists

    def save_record(self, file_name, report, extraction=None, needs_review=0):
        session = self.Session()
        try:
            new_entry = LegalAuditDB(
                file_name=file_name, category=report.category, confidence=report.confidence,
                reasoning=report.reasoning, needs_review=needs_review,
                customer_name=extraction.customer_name if extraction else "N/A",
                extracted_date=extraction.date_on_document if extraction else "N/A",
                account_id=str(extraction.account_id) if extraction else "N/A"
            )
            session.add(new_entry)
            session.commit()
        except Exception as e:
            session.rollback()
            print(f"DB Save Error: {e}")
        finally: session.close()

def get_document_text(file_path):
    """Solves the 'Poppler/PATH' issue by pointing directly to the binaries."""
    # UPDATE THIS PATH to where you extracted poppler
    POPPLER_PATH = r"C:\Program Files\poppler-25.12.0\Library\bin" 
    pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
    
    try:
        reader = pypdf.PdfReader(file_path)
        text = "\n".join([p.extract_text() for p in reader.pages if p.extract_text()])
        
        if len(text.strip()) < 150:
            print(f"🔍 OCR-ing scanned PDF: {os.path.basename(file_path)}")
            images = convert_from_path(file_path, poppler_path=POPPLER_PATH) 
            text = "\n".join([pytesseract.image_to_string(img) for img in images])
        return text
    except Exception as e:
        print(f"Read Error {file_path}: {e}")
        return ""

def clean_ai_response(raw_content):
    """
    HEALING MIDDLEWARE:
    Fixes the common issue where 8B models return 'reasoning' as a list 
    instead of a string, which triggers Pydantic validation errors.
    """
    try:
        data = json.loads(raw_content)
        
        # Self-healing logic for list vs string errors
        for key in ["reasoning", "customer_name", "summary", "category"]:
            if key in data and isinstance(data[key], list):
                data[key] = " ".join(str(item) for item in data[key])
        
        return data
    except Exception as e:
        print(f"JSON Parse/Heal Error: {e}")
        return raw_content

def call_ai_with_retry(prompt, is_extraction=False):
    """
    Tiered Retry with JSON Healing.
    Tier 1: Primary (70B)
    Tier 2: Instant (8B) - Higher Rate Limits
    Tier 3: Hail Mary (70B after Cooldown)
    """
    
    def attempt_completion(model_name):
        response = completion(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0
        )
        raw_content = response.choices[0].message.content
        # Heal data before Pydantic validation
        healed_data = clean_ai_response(raw_content)
        
        if is_extraction:
            return ExtractionReport(**healed_data)
        return DocReport(**healed_data)

    # --- TIER 1 ---
    try:
        return attempt_completion(PRIMARY_MODEL)
    
    except RateLimitError:
        print(f"⏳ {PRIMARY_MODEL} rate limited. Switching to {INSTANT_MODEL} instantly...")
        
        # --- TIER 2 ---
        try:
            return attempt_completion(INSTANT_MODEL)
        
        except RateLimitError:
            print(f"{INSTANT_MODEL} also throttled. Entering 20s cooldown...")
            
            # --- TIER 3 ---
            time.sleep(20) 
            return attempt_completion(PRIMARY_MODEL)

    except Exception as e:
        print(f"Critical Error: {e}")
        raise e

In [47]:
# Initialize Objects
db_agent = DatabaseAgent()
ARCHIVE_LOG = "archived_documents.txt"
CSV_FILE = "legal_processing_results.csv"
HITL_FILE = "human_review_list.csv"  # Dedicated file for your reference

def run_robust_batch():
    pdf_files = glob.glob("./data/*.pdf")
    print(f"Batch started for {len(pdf_files)} files...")

    # Ensure the HITL file exists and has headers if it's new
    if not os.path.exists(HITL_FILE):
        pd.DataFrame(columns=["File", "Category", "Confidence", "Reasoning", "Timestamp"]).to_csv(HITL_FILE, index=False)

    for file_path in pdf_files:
        file_name = os.path.basename(file_path)
        
        # 1. SKIP LOGIC (Token Saver)
        if db_agent.is_already_processed(file_name):
            print(f"Skipping {file_name} (Already in DB)")
            continue

        try:
            # 2. OCR / TEXT EXTRACTION
            text = get_document_text(file_path)
            if not text: 
                print(f"No text found in {file_name}. Skipping AI.")
                continue
            
            # 3. HIGH-CONTRAST CLASSIFICATION
            class_prompt = f"""
            Analyze this legal text. Return JSON with 'category', 'confidence', 'reasoning'.
            RULES:
            1. 'Irrelevant': Use for Settlements, POA, Estate docs, or Informational letters.
            2. 'Uncertain': Use for Debt Verification, Identity Validation, or Vague 'Pauses'.
            3. 'Cease': Use for explicit "Stop contacting/Stop activity" commands.
            Text: {text[:5000]}
            """
            
            report = call_ai_with_retry(class_prompt)
            ext = None

            # 4. DECISION MATRIX & HITL LOGIC
            
            # --- BRANCH A: HUMAN IN THE LOOP (Low Confidence or Uncertain) ---
            if report.category == "Uncertain" or report.confidence < 0.80:
                db_agent.save_record(file_name, report, needs_review=1)
                status = "NEEDS_HUMAN_REVIEW"
                
                # Append to the separate HITL reference file
                with open(HITL_FILE, 'a', newline='', encoding='utf-8') as f:
                    writer = csv.writer(f)
                    writer.writerow([
                        file_name, 
                        report.category, 
                        report.confidence, 
                        report.reasoning, 
                        datetime.datetime.now()
                    ])
                
            # --- BRANCH B: AUTOMATIC ARCHIVE (Settlements/POAs) ---
            elif report.category == "Irrelevant":
                with open(ARCHIVE_LOG, "a") as f: 
                    f.write(f"{datetime.datetime.now()} | {file_name}\n")
                db_agent.save_record(file_name, report, needs_review=0)
                status = "ARCHIVED"
                
            # --- BRANCH C: AUTOMATIC EXTRACTION (High-confidence Cease) ---
            else: 
                extract_prompt = f"Extract into JSON: 'customer_name', 'date_on_document', 'account_id', 'summary'. Text: {text[:5000]}"
                ext = call_ai_with_retry(extract_prompt, is_extraction=True)
                db_agent.save_record(file_name, report, extraction=ext, needs_review=0)
                status = "SAVED_TO_DB"

            print(f"{file_name} -> {status} ({report.category} - {report.confidence})")

        except Exception as e:
            print(f"Critical Failure on {file_name}: {e}")

    # --- FINAL STEP: MASTER CSV EXPORT ---
    print(f"\n📊 Exporting current database state to {CSV_FILE}...")
    try:
        engine = create_engine('sqlite:///legal_docs.db')
        df = pd.read_sql("SELECT * FROM cease_document_logs", engine)
        df.to_csv(CSV_FILE, index=False)
        print(f"Master CSV updated. Total records: {len(df)}")
        
        # Quick summary for the developer
        hitl_count = len(pd.read_csv(HITL_FILE)) if os.path.exists(HITL_FILE) else 0
        print(f"{hitl_count} items currently in {HITL_FILE} for your review.")
        
    except Exception as e:
        print(f"Failed to export CSV: {e}")

run_robust_batch()

Batch started for 1 files...
Skipping 07_software_license_violation.pdf (Already in DB)

📊 Exporting current database state to legal_processing_results.csv...
Master CSV updated. Total records: 28
3 items currently in human_review_list.csv for your review.


In [ ]:
def clear_all_records(db_agent):
    session = db_agent.Session()
    try:
        # This deletes every row in the table
        num_rows_deleted = session.query(LegalAuditDB).delete()
        session.commit()
        print(f"Deleted {num_rows_deleted} records from the database.")
    except Exception as e:
        session.rollback()
        print(f"Failed to clear records: {e}")
    finally:
        session.close()

# Execute the clear
clear_all_records(db_agent)

csv_file = "legal_processing_results.csv"
if os.path.exists(csv_file):
    os.remove(csv_file)
    print(f"📄 CSV backup {csv_file} deleted.")

🧹 Deleted 0 records from the database.
